# Trimorious 2.0 | M2.3a — Paired scientific evaluation and optimization decision

**Notebook 05 of 07 · GPU runtime**

**Use a clean T4 GPU runtime.** Requires notebook 04's final adapter. Save `trimorious_m1_3_2_PRIVATE_full_1c3d70c27cc3c9f8.zip` to the project folder. Supply a separate externally authored and expert-key-reviewed fresh JSONL (at least 100 objective tasks, 10 templates, unique IDs/questions; fields `id`, `question`, `template`, `family`, `kind` = numeric or choice, `gold`, and numeric `atol`, `rtol`). Choice options A–D must appear in question. Supply `FRESH_TEST_PATH`, `FRESH_TEST_SHA256` (externally sealed hash), and `FRESH_INDEPENDENCE_ATTESTED=True` only after verifying independence. If a fresh hash was frozen during notebook 02, use that hash. Do not allow the test author to see the adapter results or tune on the fresh benchmark.

This runs the original hardened D07 scoring protocol for 1,000 private diagnostic tasks plus the fresh test, with matched inference settings for base and adapter. SQLite checkpoints are committed per response; interrupted evaluation resumes. The complete paired run may consume substantial GPU quota. No test keys are published. After evaluation, fill the blinded `rubric_review_private.csv` (120 tasks; `a_acceptable`, `b_acceptable` yes/no, reviewer and notes). Keep the private mapping hidden from reviewer. Inspect `evaluation_report.json`. If the objective gate fails, do not publish. DPO/GRPO are optional research interventions only if supported by separate validated preference/reward data and a new development evaluation plan. Do not tune on the fresh test or weaken gates; if you change the adapter, a new sealed confirmation test is necessary. No optional training implemented in this sequence.

All stages use the same `MyDrive/trimorious_PRIVATE_m1_4_1/release_v2/` workspace. Save this notebook in Drive. Edit only the configuration and select Runtime → Run all. Never run another GPU notebook concurrently. Each notebook is self-contained and contains no credentials. GPU execution is not locally tested.

In [ ]:
# @title Configuration — EVALUATE
STAGE = 'EVALUATE'
USE_DRIVE = True # @param {type:'boolean'}
INSTALL_GPU_DEPS = True # @param {type:'boolean'}
PROJECT_ROOT = '/content/drive/MyDrive/trimorious_PRIVATE_m1_4_1'
SOURCE_NAME = 'trimorious_m1_4_1_PRIVATE_GPU_partial_20261127T061605946898Z.zip'
D07_FILENAME = 'trimorious_m1_3_2_PRIVATE_full_1c3d70c27cc3c9f8.zip'
# Save this benchmark ZIP in PROJECT_ROOT (or upload to /content once). Required for EVALUATE.
FRESH_TEST_PATH = '' # @param {type:'string'}
# Fresh test JSONL format documented in section 5. Must be externally authored, sealed before evaluation; preferably before training.
FRESH_TEST_SHA256 = '' # @param {type:'string'}
FRESH_INDEPENDENCE_ATTESTED = False # @param {type:'boolean'}
# Confirm external authorship, expert key review, no access to this adapter's evaluation outputs, no tuning on the test.
DATA_LICENSE_REVIEWED = False # @param {type:'boolean'}
DATA_REVIEW_APPROVED = False # @param {type:'boolean'}
# By approving, you confirm rights to train and distribute the resulting adapter, and actual review of CSV rows.
HUMAN_EVAL_APPROVED = False # @param {type:'boolean'}
# After actual rubric review; see evaluation report. Not an automatic pass.
PUBLISH_APPROVAL = '' # @param {type:'string'}
MAKE_PUBLIC = False # @param {type:'boolean'}
PUBLIC_APPROVAL = '' # @param {type:'string'}
MAX_TRAIN_PER_TEMPLATE = 12
MAX_VAL_PER_TEMPLATE = 6
MAX_LENGTH = 512
GRAD_ACCUM = 16
EPOCHS = 1
SEED = 417031
# Provisional minimum release policy: approve before training; cannot change after dataset freeze.
MIN_D07_GAIN = 1.0       # percentage points, measurable diagnostic gain
MAX_FAMILY_DROP = 5.0   # percentage points
MIN_FRESH_GAIN = 0.0
MAX_PROTOCOL_DROP = 2.0
BASE = 'Qwen/Qwen3-4B-Instruct-2507'
REV = 'cdbee75f17c01a7cc42f958dc650907174af0554'
TARGET = 'roime81/trimorious-research-4B-v2'
EXPECTED_CORPUS = '9fe5e9dbce17096c4d1ddd28a82e45111a37cbba66c574ec63473f6f70dd283a'

In [ ]:
# @title Mount Drive and prepare local worker scripts
from pathlib import Path
import json, os, sys, subprocess, hashlib, csv, sqlite3, shutil, traceback
from collections import Counter
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
ROOT = Path(PROJECT_ROOT)
assert ROOT.is_dir(), f'Project folder missing: {ROOT}'
WORK = ROOT/'release_v2'; WORK.mkdir(exist_ok=True)
SCRIPT_DIR = Path('/tmp/trimorious_v2_sequence'); SCRIPT_DIR.mkdir(exist_ok=True)
print('Persistent workspace:', WORK)
print('Local script directory:', SCRIPT_DIR)
CONFIG_NAMES=['USE_DRIVE', 'INSTALL_GPU_DEPS', 'PROJECT_ROOT', 'SOURCE_NAME', 'D07_FILENAME', 'FRESH_TEST_PATH', 'FRESH_TEST_SHA256', 'FRESH_INDEPENDENCE_ATTESTED', 'DATA_LICENSE_REVIEWED', 'DATA_REVIEW_APPROVED', 'HUMAN_EVAL_APPROVED', 'PUBLISH_APPROVAL', 'MAKE_PUBLIC', 'PUBLIC_APPROVAL', 'MAX_TRAIN_PER_TEMPLATE', 'MAX_VAL_PER_TEMPLATE', 'MAX_LENGTH', 'GRAD_ACCUM', 'EPOCHS', 'SEED', 'MIN_D07_GAIN', 'MAX_FAMILY_DROP', 'MIN_FRESH_GAIN', 'MAX_PROTOCOL_DROP', 'BASE', 'REV', 'TARGET', 'EXPECTED_CORPUS']
CONFIG={k:globals()[k] for k in CONFIG_NAMES}
assert TARGET=='roime81/trimorious-research-4B-v2' and BASE=='Qwen/Qwen3-4B-Instruct-2507' and REV=='cdbee75f17c01a7cc42f958dc650907174af0554'

## Embedded implementation
The following cells write the implementation to the local runtime. Run them unchanged. No source code download required.

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/common.py
# @title 1. Mount Drive and common helpers (CPU)
import os, sys, json, hashlib, math, re, random, csv, zipfile, sqlite3, statistics, time, gc, shutil, platform
from pathlib import Path
from collections import Counter, defaultdict
from fractions import Fraction
from importlib import metadata
assert STAGE in {'RECOVER','APPROVE','SMOKE','TRAIN','EVALUATE','PACKAGE','PUBLISH'}
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
ROOT=Path(PROJECT_ROOT); assert ROOT.is_dir(), f'Missing project folder: {ROOT}'
WORK=ROOT/'release_v2'; WORK.mkdir(exist_ok=True)
def canon(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'),allow_nan=False)
def sha(b): return hashlib.sha256(b if isinstance(b,bytes) else canon(b).encode()).hexdigest()
def put(p,obj):
    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True); tmp=p.with_suffix(p.suffix+'.tmp'); tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,allow_nan=False)); tmp.replace(p)
def get(p): return json.loads(Path(p).read_text())
def lines(b): return [json.loads(x) for x in b.splitlines() if x.strip()]
def readl(p): return lines(Path(p).read_text())
def writel(p,rows):
    p=Path(p); tmp=p.with_suffix(p.suffix+'.tmp'); tmp.write_text(''.join(canon(r)+'\n' for r in rows)); tmp.replace(p)
def writecsv(p,rows,fields):
    with Path(p).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(rows)
def archive_check(z):
    ns=z.namelist(); assert len(ns)==len(set(ns)) and z.testzip() is None
    m=json.loads(z.read('archive_manifest.json')); assert set(m)==set(ns)-{'archive_manifest.json'}
    assert all(sha(z.read(k))==v for k,v in m.items()), 'Archive hash mismatch'
    return m
def state(stage,**kw):
    p=WORK/'project_state.json'; s=get(p) if p.exists() else {}; s[stage]=kw; put(p,s)
def norm(s): return re.sub(r'\s+',' ',s.casefold()).strip()
def mask(s): return re.sub(r'(?<![a-z])[-+]?\d+(?:\.\d+)?(?:e[-+]?\d+)?','NUM',norm(s))
def close(a,b): return math.isclose(float(a),float(b),rel_tol=1e-8,abs_tol=1e-10)
POLICY={'min_d07_gain_pp':MIN_D07_GAIN,'max_family_drop_pp':MAX_FAMILY_DROP,'min_fresh_gain_pp':MIN_FRESH_GAIN,'max_protocol_drop_pp':MAX_PROTOCOL_DROP}
print('Stage:',STAGE,'| Persistent work directory:',WORK)

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/approve.py
def load_approved():
    approval=get(WORK/'dataset_approval.json'); assert approval['training_approved'] is True
    for name,digest in approval['files'].items(): assert sha((WORK/name).read_bytes())==digest, f'Changed approved file: {name}'
    assert approval['policy']==POLICY and approval['base']==BASE and approval['revision']==REV
    return approval

if STAGE=='APPROVE':
    assert DATA_LICENSE_REVIEWED and DATA_REVIEW_APPROVED, 'Review rights and examples first; then set both approval flags.'
    report=get(WORK/'recovery.json'); candidates=readl(WORK/'review_candidates.jsonl'); assert sha(candidates)==report['review_set_sha256']
    with (WORK/'review_queue.csv').open(newline='',encoding='utf-8-sig') as f: reviews=list(csv.DictReader(f))
    by_id={r['id']:r for r in reviews}; assert len(reviews)==len(by_id)==len(candidates) and set(by_id)=={r['id'] for r in candidates}
    approved={'train':[],'validation':[]}; dispositions=[]
    for r in candidates:
        q=by_id[r['id']]; assert q['prompt']==r['prompt'] and q['response']==r['reference_response'] and q['split']==r['split'] and q['template']==r['template_id']
        d=q['decision'].strip().lower(); assert d in {'','approve','reject'}
        if d: assert q['reviewer'].strip() and q['notes'].strip(), 'Each explicit decision requires reviewer and rationale.'
        dispositions.append({'id':r['id'],'decision':d or 'unreviewed','reviewer':q['reviewer'],'notes':q['notes']})
        if d=='approve':
            assert close(recompute(r),r['numeric_answer']) and not r['d07_screen']['flagged']
            # Use the source response as written, not unapproved teacher output.
            split='train' if r['split']=='development_candidate' else 'validation'
            approved[split].append({'id':r['id'],'template':r['template_id'],'family':r['family'],'messages':[{'role':'user','content':r['prompt']},{'role':'assistant','content':r['reference_response']}]})
    assert len(approved['train'])>=32 and len(approved['validation'])>=8, 'Review at least 32 development and 8 validation examples; these are smoke-level minima, not a quality guarantee.'
    assert {r['template'] for r in approved['train']}.isdisjoint({r['template'] for r in approved['validation']})
    assert not ({mask(r['messages'][0]['content']) for r in approved['train']} & {mask(r['messages'][0]['content']) for r in approved['validation']})
    if FRESH_TEST_SHA256: assert re.fullmatch('[a-f0-9]{64}',FRESH_TEST_SHA256)
    data_names=['train.jsonl','validation.jsonl','review_dispositions.jsonl']
    contents=[approved['train'],approved['validation'],dispositions]
    files={name:sha(''.join(canon(r)+'\n' for r in rows).encode()) for name,rows in zip(data_names,contents)}
    approval={'training_approved':True,'license_review_attested':True,'corpus_sha256':EXPECTED_CORPUS,'review_set_sha256':report['review_set_sha256'],'files':files,'counts':{k:len(v) for k,v in approved.items()},'policy':POLICY,'fresh_test_sha256':FRESH_TEST_SHA256,'base':BASE,'revision':REV,'scope':'human-reviewed, recomputed synthetic math/science/statistics; no teacher or tool trace SFT'}
    if (WORK/'dataset_approval.json').exists(): assert get(WORK/'dataset_approval.json')==approval, 'Approval already frozen. New revision requires separate workspace.'
    for name,rows in zip(data_names,contents): writel(WORK/name,rows)
    put(WORK/'dataset_approval.json',approval); state('APPROVE',complete=True,counts=approval['counts']); print(json.dumps(approval,indent=2)); print('NEXT: SMOKE')

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/gpu.py
if STAGE in {'SMOKE','TRAIN','EVALUATE'}:
    import subprocess
    if INSTALL_GPU_DEPS:
        lock_path=WORK/'auto_gpu_versions.json'
        if lock_path.exists():
            locked=get(lock_path)
            assert metadata.version('torch')==locked['torch'], 'PyTorch changed in this runtime. Use a compatible runtime; do not mix training checkpoints.'
            specs=[k+'=='+v for k,v in locked.items() if k!='torch']+['tqdm','scipy','scikit-learn']
        else:
            specs=['transformers==4.57.6','peft>=0.17,<0.19','accelerate>=1.6,<2','bitsandbytes>=0.48,<1','huggingface_hub>=0.34,<1','safetensors>=0.5','tqdm','scipy','scikit-learn']
        subprocess.check_call([sys.executable,'-m','pip','install','-q',*specs])
    import torch
    from transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig,Trainer,TrainingArguments,TrainerCallback,set_seed
    from transformers.trainer_utils import get_last_checkpoint
    from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training,PeftModel
    from tqdm.auto import tqdm
    assert torch.cuda.is_available(), 'Select a GPU runtime.'
    versions={x:metadata.version(x) for x in ['torch','transformers','peft','accelerate','bitsandbytes','huggingface_hub','safetensors']}
    lock_path=WORK/'auto_gpu_versions.json'
    if lock_path.exists(): assert get(lock_path)==versions, 'GPU dependency versions changed; stop rather than mix checkpoints.'
    else: put(lock_path,versions)
    print('GPU:',torch.cuda.get_device_name(0),'Versions:',versions,flush=True)
    def load_base(train=False):
        gc.collect(); torch.cuda.empty_cache(); free,total=torch.cuda.mem_get_info()
        print(f'Free GPU: {free/2**30:.2f}/{total/2**30:.2f} GiB',flush=True)
        assert free/2**30>=10, 'Not enough free GPU memory. Use a clean runtime; do not load teacher models.'
        tok=AutoTokenizer.from_pretrained(BASE,revision=REV,trust_remote_code=False)
        assert tok.chat_template
        if tok.pad_token_id is None: tok.pad_token=tok.eos_token
        tok.padding_side='right'
        quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
        mdl=AutoModelForCausalLM.from_pretrained(BASE,revision=REV,quantization_config=quant,device_map={'':0},torch_dtype=torch.float16,attn_implementation='eager',low_cpu_mem_usage=True,trust_remote_code=False,use_safetensors=True)
        assert getattr(mdl,'is_loaded_in_4bit',False)
        mdl.config.use_cache=not train
        if train:
            mdl=prepare_model_for_kbit_training(mdl,use_gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False})
            mdl=get_peft_model(mdl,LoraConfig(r=8,lora_alpha=16,lora_dropout=.05,bias='none',task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],revision=REV))
            mdl.print_trainable_parameters(); assert all('lora_' in name for name,p in mdl.named_parameters() if p.requires_grad)
        return mdl,tok
    class Rows(torch.utils.data.Dataset):
        def __init__(self,data): self.data=data
        def __len__(self): return len(self.data)
        def __getitem__(self,i): return self.data[i]
    def tokenize_rows(rows,tok):
        result=[]; too_long=[]
        for r in rows:
            msgs=r['messages']; prefix=tok.apply_chat_template(msgs[:-1],tokenize=True,add_generation_prompt=True); full=tok.apply_chat_template(msgs,tokenize=True,add_generation_prompt=False)
            assert full[:len(prefix)]==prefix, f'Chat template prefix mismatch: {r["id"]}'
            assert len(full)>len(prefix) and tok.eos_token_id in full[len(prefix):], f'No supervised EOS: {r["id"]}'
            if len(full)>MAX_LENGTH: too_long.append(r['id']); continue
            labels=[-100]*len(prefix)+full[len(prefix):]; assert all(x==-100 for x in labels[:len(prefix)]) and any(x!=-100 for x in labels[len(prefix):])
            result.append({'input_ids':full,'attention_mask':[1]*len(full),'labels':labels})
        assert not too_long, f'Would truncate {len(too_long)} answers; revise config before smoke: {too_long[:5]}'
        print('Tokenized',len(result),'max length',max(len(r['input_ids']) for r in result)); return Rows(result)
    def collate(items):
        n=max(len(r['input_ids']) for r in items); out={}
        for key,pad in [('input_ids',tokenizer.pad_token_id),('attention_mask',0),('labels',-100)]: out[key]=torch.tensor([r[key]+[pad]*(n-len(r[key])) for r in items],dtype=torch.long)
        return out
    class Health(TrainerCallback):
        def __init__(self): self.nonzero=False; self.logs=[]
        def on_pre_optimizer_step(self,args,state,control,model=None,**kw):
            grads=[p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
            assert grads and all(torch.isfinite(g).all().item() for g in grads), 'Missing or nonfinite adapter gradients'
            self.nonzero |= any(torch.count_nonzero(g).item()>0 for g in grads)
        def on_log(self,args,state,control,logs=None,**kw):
            if logs:
                for k in ('loss','eval_loss','train_loss','grad_norm'):
                    if k in logs: assert math.isfinite(float(logs[k])),f'Nonfinite {k}'
                row={**logs,'step':state.global_step,'peak_gpu_gib':torch.cuda.max_memory_allocated()/2**30}; self.logs.append(row); print(row,flush=True)
    def training_config(approval): return {'approval_sha256':sha(approval),'base':BASE,'revision':REV,'max_length':MAX_LENGTH,'grad_accum':GRAD_ACCUM,'epochs':EPOCHS,'seed':SEED,'lora_rank':8,'alpha':16,'lr':1e-4,'precision':'fp16_nf4','versions':versions}

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/eval_setup.py
def active_run():
    approval=load_approved(); p=WORK/'sft_output'/get(WORK/'active_run.json')['run']; done=get(p/'training_complete.json')
    assert done['config']['approval_sha256']==sha(approval)
    final=p/'final_adapter'
    assert all(sha((final/k).read_bytes())==v for k,v in done['adapter_hashes'].items()), 'Adapter changed'
    return approval,p,done,final

if STAGE=='EVALUATE':
    approval,run_dir,training_done,final=active_run()
    assert not (approval['fresh_test_sha256'] or (run_dir/'fresh_test_seal.json').exists()) or FRESH_TEST_PATH, 'A sealed fresh test is configured. Provide its path to avoid repeating D07 evaluation.'
    d07=ROOT/D07_FILENAME
    if not d07.exists(): d07=Path('/content')/D07_FILENAME
    assert d07.is_file(), 'Upload original M1.3.2 full archive.'
    with zipfile.ZipFile(d07) as z:
        archive_check(z); pre='PRIVATE_DO_NOT_TRAIN/'; m=json.loads(z.read(pre+'manifest.json'))
        assert m['training_use'] is False and all(sha(z.read(pre+k))==v for k,v in m['file_hashes'].items())
        questions=lines(z.read(pre+'D07_questions.jsonl')); keys=lines(z.read(pre+'D07_gold_key.jsonl')); keymap={r['id']:r for r in keys}
        assert len(questions)==len(keys)==len(keymap)==1000
        assert all(all(keymap[q['id']].get(k,v)==v for k,v in q.items()) for q in questions)
        benchmark=[{**q,**keymap[q['id']]} for q in questions]
        assert sha(benchmark)==m['benchmark_sha256']=='47694f1b2f4dbd44e8f5b6e58962ad07af5c2a844e375249e74d785de0b51283'
        assert sha(z.read(pre+'D07_questions.jsonl'))=='e724980d46b644f47952d3ab87789e94a32cb0b79835c4721ad3c7616c311c7e'
    fresh=[]
    if FRESH_TEST_PATH:
        fresh_bytes=Path(FRESH_TEST_PATH).read_bytes(); fresh_digest=sha(fresh_bytes)
        assert FRESH_INDEPENDENCE_ATTESTED, 'Confirm independent authorship and answer-key review before evaluation.'
        expected=approval['fresh_test_sha256'] or FRESH_TEST_SHA256
        assert expected and fresh_digest==expected, 'Fresh test hash missing or mismatch.'
        assert not approval['fresh_test_sha256'] or not FRESH_TEST_SHA256 or FRESH_TEST_SHA256==expected
        seal={'sha256':fresh_digest,'adapter_hashes':training_done['adapter_hashes'],'independence_attested':True,'sealed_before_training':bool(approval['fresh_test_sha256'])}
        seal_path=run_dir/'fresh_test_seal.json'
        if seal_path.exists(): assert get(seal_path)==seal, 'Fresh confirmation already sealed for this frozen adapter.'
        fresh=lines(fresh_bytes); assert len(fresh)>=100 and len({r['template'] for r in fresh})>=10
        assert len({r['id'] for r in fresh})==len(fresh) and len({norm(r['question']) for r in fresh})==len(fresh)
        for r in fresh:
            assert r['kind'] in {'numeric','choice'} and all(isinstance(r[k],str) and r[k].strip() for k in ['id','question','template','family'])
            if r['kind']=='numeric': assert isinstance(r['gold'],(int,float)) and not isinstance(r['gold'],bool) and math.isfinite(r['gold']) and 0<=r['atol']<=.01 and 0<=r['rtol']<=.001
            else: assert r['gold'] in 'ABCD' and len(r['gold'])==1
        known=[r['messages'][0]['content'] for name in ['train.jsonl','validation.jsonl'] for r in readl(WORK/name)]+[r['question'] for r in benchmark]
        assert not ({mask(q) for q in known}&{mask(r['question']) for r in fresh}), 'Fresh masked question overlap'
        assert not ({r['id'] for r in fresh}&{r['id'] for r in benchmark})
        # Check lexical similarity without using gold. This is not semantic independence proof.
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity
        v=TfidfVectorizer(analyzer='char',ngram_range=(3,5)); x=v.fit_transform(known+[r['question'] for r in fresh]); sim=cosine_similarity(x[len(known):],x[:len(known)]).max()
        assert sim<.78,f'Fresh test near-overlap: {sim}'
        put(seal_path,seal)
    evaluation_config={'base':BASE,'revision':REV,'adapter_hashes':training_done['adapter_hashes'],'versions':versions,'benchmark_hash':sha(benchmark),'fresh_hash':fresh_digest if fresh else '', 'max_new_tokens':384,'seed':132,'attention':'eager','precision':'nf4_fp16','scorer':'v0.2'}
    eval_dir=run_dir/'evaluation'/sha(evaluation_config)[:20]; eval_dir.mkdir(parents=True,exist_ok=True); put(eval_dir/'config.json',evaluation_config)
    print('Evaluation rows:',len(benchmark),'+',len(fresh),'fresh; checkpoints:',eval_dir)

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/scorer.py
if STAGE=='EVALUATE':
    def options(t): return dict(re.findall(r'^([ABCD])\. (.+)$',t['question'],re.M))

    def last_object(text):
        # Parse final JSON object (including nested tool args); never eval model output.
        text=str(text).strip()
        if len(text)>20000: return None
        def reject_nonfinite(s): raise ValueError('Non-finite JSON constant')
        dec=json.JSONDecoder(parse_constant=reject_nonfinite); candidates=[]
        for m in re.finditer(r'\{',text):
            try:
                obj,end=dec.raw_decode(text[m.start():]); candidates.append((m.start()+end,m.start(),obj))
            except (ValueError,RecursionError): pass
        if not candidates: return None
        _,_,obj=max(candidates,key=lambda x:(x[0],-x[1]))
        if not isinstance(obj,dict): return None
        try: canon(obj)
        except (ValueError,TypeError,OverflowError): return None
        return obj

    def numeric_ok(x,gold,atol,rtol):
        if isinstance(x,bool) or not isinstance(x,(int,float)): return False
        try: return math.isfinite(float(x)) and math.isclose(float(x),gold,abs_tol=atol,rel_tol=rtol)
        except (OverflowError,ValueError): return False

    def execute_tool(task, raw):
        obj=last_object(raw)
        if not obj or set(obj)!={'tool','args'} or obj['tool']!=task['tool'] or canon(obj['args'])!=canon(task['tool_args']):
            return {'ok':False,'error':'Invalid or incorrect tool call; no tool executed.'}
        args=obj['args']; name=obj['tool']
        # Exact argument matching limits execution to the small authored fixture for each task.
        if name=='mean': val=statistics.mean(args['values'])
        elif name=='sample_variance': val=statistics.variance(args['values'])
        elif name=='dot': val=sum(x*y for x,y in zip(args['a'],args['b']))
        else: return {'ok':False,'error':'Unknown tool'}
        return {'ok':True,'value':val}

    def score(task, raw, tool_result=None):
        obj=last_object(raw); protocol=bool(obj is not None and set(obj)=={'answer'})
        val=obj.get('answer') if protocol else None
        kind=task['kind']; correct=None
        if kind=='rubric': protocol=bool(protocol and isinstance(val,str) and val.strip())
        if kind=='choice': correct=bool(protocol and isinstance(val,str) and val.strip().upper()==task['gold'])
        elif kind in ('numeric','tool_numeric'):
            correct=bool(protocol and numeric_ok(val,task['gold'],task['atol'],task['rtol']))
            if kind=='tool_numeric': correct=bool(correct and tool_result and tool_result.get('ok'))
        return {'score':None if correct is None else int(correct),'protocol_ok':protocol,'parsed_answer':val,
                'review_status':'human_review_required' if kind=='rubric' else 'objective_scored'}

    def prompt_for(task):
        if task['kind']=='tool_numeric':
            return task['question']+'\nAvailable tools: mean(values: list of numbers), sample_variance(values: list of numbers), dot(a: list of numbers, b: list of numbers). First return only JSON {"tool":"tool_name","args":{...}} with the required arguments. You will receive a tool result next.'
        fmt={'numeric':'a JSON number (not a quoted string); give at least six significant digits if not exact', 'choice':'the single choice letter A, B, C or D', 'rubric':'your concise written response as a JSON string'}[task['kind']]
        return task['question']+'\nReturn only a JSON object with key "answer" containing '+fmt+'.'

    legacy_score=score
    NUM=r'[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?'
    def as_number(v):
        if isinstance(v,dict) and set(v)=={'value'}: v=v['value']
        if isinstance(v,bool): return None
        if isinstance(v,(float,int)):
            try: return float(v) if math.isfinite(float(v)) else None
            except OverflowError: return None
        if not isinstance(v,str) or len(v)>100: return None
        v=v.strip()
        try:
            if re.fullmatch(NUM,v): out=float(v)
            elif re.fullmatch(r'[+-]?\d+\s*/\s*[+-]?\d+',v):
                a,b=v.split('/'); out=float(Fraction(int(a),int(b)))
            else: return None
            return out if math.isfinite(out) else None
        except (ValueError,ZeroDivisionError,OverflowError): return None

    def parse_response(raw):
        raw=str(raw).strip()
        if len(raw)>20000: return None,None,False
        def no_dup(pairs):
            d={}
            for k,v in pairs:
                if k in d: raise ValueError('duplicate key')
                d[k]=v
            return d
        def reject(s): raise ValueError('nonfinite')
        wrapped=re.fullmatch(r'```(?:json)?\s*([\s\S]*?)\s*```',raw,re.I)
        text=wrapped.group(1).strip() if wrapped else raw
        # Recover only known presentation artifacts, not arbitrary prose.
        if not wrapped:
            text=re.sub(r'^```(?:json)?\s*','',text,flags=re.I).strip()
            text=re.sub(r'^\|\s*(?=\{)','',text).strip()
        try:
            obj=json.loads(text,object_pairs_hook=no_dup,parse_constant=reject)
            if isinstance(obj,dict): return obj.get('answer') if set(obj)=={'answer'} else None,obj,text==raw
            if isinstance(obj,(str,int,float)) and not isinstance(obj,bool): return obj,None,False
        except (ValueError,RecursionError):
            if re.fullmatch(r'[A-Da-d]|'+NUM+r'|[+-]?\d+\s*/\s*[+-]?\d+',text): return text,None,False
        return None,None,False

    def hardened(t,raw,tool_result=None):
        v,obj,whole=parse_response(raw); k=t['kind']; numeric=as_number(v)
        if k=='rubric':
            protocol=bool(whole and obj and isinstance(v,str) and v.strip())
            return dict(strict_protocol=protocol,answer_parseable=bool(isinstance(v,str) and v.strip()),content_correct=None,operational_correct=None,parsed_value=None)
        if k=='choice':
            match=None
            if isinstance(v,str):
                s=v.strip(); match=s.upper() if re.fullmatch('[A-Da-d]',s) else next((a for a,b in options(t).items() if s.casefold()==b.casefold()),None)
            protocol=bool(whole and obj and isinstance(v,str) and re.fullmatch('[A-D]',v))
            parseable=match is not None; correct=match==t['gold']; val=match
        else:
            protocol=bool(whole and obj and isinstance(v,(int,float)) and not isinstance(v,bool) and numeric is not None)
            parseable=numeric is not None; correct=bool(parseable and numeric_ok(numeric,t['gold'],t['atol'],t['rtol'])); val=numeric
        operational=correct and (k!='tool_numeric' or bool(tool_result and tool_result.get('ok')))
        return dict(strict_protocol=protocol,answer_parseable=parseable,content_correct=int(correct),operational_correct=int(operational),parsed_value=val)

    for t in benchmark:
        if t['kind']=='rubric': continue
        tool=execute_tool(t,canon({'tool':t['tool'],'args':t['tool_args']})) if t['kind']=='tool_numeric' else None
        assert hardened(t,canon({'answer':t['gold']}),tool)['operational_correct']==1
        assert hardened(t,'malformed',tool)['operational_correct']==0
        if t['kind']!='choice':
            assert hardened(t,canon({'answer':{'value':str(t['gold'])}}),tool)['content_correct']==1
            assert not hardened(t,canon({'answer':{'value':str(t['gold'])}}),tool)['strict_protocol']
            assert not hardened(t,'{"answer":true}',tool)['answer_parseable']
    assert as_number('1/4')==0.25 and as_number('1/0') is None
    assert parse_response('{"answer":1,"answer":2}')[0] is None
    print('Scoring self-tests passed for',sum(t['kind']!='rubric' for t in benchmark),'objective tasks')

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/evaluate.py
if STAGE=='EVALUATE':
    SYSTEM='You are being evaluated on scientific reasoning. Use only the supplied information. Follow the specified JSON output format. Do not invent evidence.'
    MAX_NEW_TOKENS=384
    def generate(mdl,tok,messages):
        prompt=tok.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
        batch=tok(prompt,return_tensors='pt',add_special_tokens=False); batch={k:v.to(mdl.get_input_embeddings().weight.device) for k,v in batch.items()}
        torch.cuda.synchronize(); start=time.perf_counter()
        with torch.inference_mode(): out=mdl.generate(**batch,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,pad_token_id=tok.pad_token_id)
        torch.cuda.synchronize(); generated=out[0,batch['input_ids'].shape[1]:]
        return tok.decode(generated,skip_special_tokens=True),int(len(generated)),time.perf_counter()-start
    def evaluate_one(mdl,tok,t):
        messages=[{'role':'system','content':SYSTEM},{'role':'user','content':prompt_for(t)}]
        raw,n,elapsed=generate(mdl,tok,messages); first=result=None
        if t['kind']=='tool_numeric':
            first=raw; result=execute_tool(t,first)
            messages += [{'role':'assistant','content':first},{'role':'user','content':'Actual local tool execution result: '+canon(result)+'\nReturn only {"answer": numeric_value} using the observation. If execution failed, report null.'}]
            raw,n2,e2=generate(mdl,tok,messages); n+=n2; elapsed+=e2
        return {'id':t['id'],'family':t['family'],'template':t['template'],'kind':t['kind'],'response':raw,'tool_call_response':first,'tool_result':result,'generated_tokens':n,'elapsed_seconds':elapsed,**hardened(t,raw,result)}
    # One SQLite row per completed inference. No model text executed except original allowlisted tool fixture.
    db=eval_dir/'responses.sqlite3'
    with sqlite3.connect(db) as conn:
        conn.execute('CREATE TABLE IF NOT EXISTS responses (model TEXT, item TEXT, payload TEXT NOT NULL, PRIMARY KEY(model,item))'); conn.commit()
    def run_eval(label,tasks):
        with sqlite3.connect(db) as conn:
            completed={x[0] for x in conn.execute('SELECT item FROM responses WHERE model=?',(label,))}
        todo=[t for t in tasks if t['id'] not in completed]; print(label,'cached',len(completed),'pending',len(todo))
        if not todo: return
        mdl=tok=None
        try:
            set_seed(132); torch.cuda.reset_peak_memory_stats(); mdl,tok=load_base(train=False)
            if label=='trimorious_v2': mdl=PeftModel.from_pretrained(mdl,str(final),is_trainable=False)
            mdl.eval(); mdl.config.use_cache=True
            with sqlite3.connect(db) as conn:
                for t in tqdm(todo,desc=label):
                    row=evaluate_one(mdl,tok,t); row['peak_gpu_gib']=torch.cuda.max_memory_allocated()/2**30
                    conn.execute('INSERT INTO responses VALUES(?,?,?)',(label,t['id'],canon(row))); conn.commit()
        finally:
            del mdl,tok; gc.collect(); torch.cuda.empty_cache()
    tasks=benchmark+fresh
    for label in ['qwen3_base','trimorious_v2']: run_eval(label,tasks)
    with sqlite3.connect(db) as conn:
        results={label:{i:json.loads(row) for i,row in conn.execute('SELECT item,payload FROM responses WHERE model=?',(label,))} for label in ['qwen3_base','trimorious_v2']}
    assert all(set(v)=={t['id'] for t in tasks} for v in results.values())
    from scipy.stats import binomtest
    def comparison(tasks):
        obj=[t for t in tasks if t['kind']!='rubric']; a=[results['qwen3_base'][t['id']]['operational_correct'] for t in obj]; b=[results['trimorious_v2'][t['id']]['operational_correct'] for t in obj]
        n=len(a); wins=sum(x<y for x,y in zip(a,b)); losses=sum(x>y for x,y in zip(a,b)); groups=defaultdict(list)
        for t,x,y in zip(obj,a,b): groups[t['template']].append(y-x)
        rng=random.Random(132); clusters=list(groups.values()); draws=[]
        for _ in range(2000):
            sample=rng.choices(clusters,k=len(clusters)); draws.append(100*sum(map(sum,sample))/sum(map(len,sample)))
        draws.sort()
        return {'n':n,'base_pct':100*sum(a)/n,'v2_pct':100*sum(b)/n,'delta_pp':100*(sum(b)-sum(a))/n,'wins':wins,'losses':losses,'mcnemar_exact_p':float(binomtest(wins,wins+losses).pvalue) if wins+losses else 1.0,'template_bootstrap_95_ci':[draws[50],draws[1949]],'templates':len(clusters)}
    diag=comparison(benchmark); fam={f:comparison([t for t in benchmark if t['family']==f]) for f in sorted({t['family'] for t in benchmark if t['kind']!='rubric'})}; confirmation=comparison(fresh) if fresh else None
    protocol={label:100*sum(v[t['id']]['strict_protocol'] for t in benchmark)/len(benchmark) for label,v in results.items()}
    metrics={label:{'mean_latency_s':statistics.mean(v[t['id']]['elapsed_seconds'] for t in tasks),'mean_generated_tokens':statistics.mean(v[t['id']]['generated_tokens'] for t in tasks),'peak_gpu_gib':max(r['peak_gpu_gib'] for r in v.values())} for label,v in results.items()}
    checks={'d07_gain':diag['delta_pp']>=POLICY['min_d07_gain_pp'],'family_regression':all(x['delta_pp']>=-POLICY['max_family_drop_pp'] for x in fam.values()),'protocol_regression':protocol['trimorious_v2']-protocol['qwen3_base']>=-POLICY['max_protocol_drop_pp'],'fresh_confirmation':bool(confirmation and confirmation['delta_pp']>=POLICY['min_fresh_gain_pp'])}
    # Paired blinded rubric queue, stored privately. Mapping never published.
    queue=[]; mapping={}; rng=random.Random(132)
    for t in benchmark:
        if t['kind']!='rubric': continue
        labels=['qwen3_base','trimorious_v2']; rng.shuffle(labels); mapping[t['id']]=labels
        queue.append({'id':t['id'],'question':t['question'],'rubric':canon(t.get('rubric',t.get('gold'))),'answer_a':results[labels[0]][t['id']]['response'],'answer_b':results[labels[1]][t['id']]['response'],'a_acceptable':'','b_acceptable':'','reviewer':'','notes':''})
    qpath=eval_dir/'rubric_review_private.csv'
    if not qpath.exists(): writecsv(qpath,queue,list(queue[0]))
    put(eval_dir/'rubric_mapping_private.json',mapping)
    report={'evaluation_config':evaluation_config,'d07':diag,'families':fam,'fresh':confirmation,'protocol_pct':protocol,'runtime_metrics':metrics,'checks':checks,'objective_gate_passed':all(checks.values()),'rubric_pending':len(queue),'note':'D07 diagnostic, not independent; fresh set author and scientific validity require human attestation. Cluster intervals are descriptive, not proof.'}
    put(eval_dir/'evaluation_report.json',report); put(WORK/'active_evaluation.json',{'run':run_dir.name,'evaluation':eval_dir.name}); state('EVALUATE',complete=True,objective_gate=all(checks.values()),report=str(eval_dir/'evaluation_report.json')); print(json.dumps(report,indent=2)); print('NEXT: finish private rubric review before PACKAGE. If gate fails, do not publish.')

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/worker.py
import sys, json
from pathlib import Path
SCRIPTS=Path(__file__).resolve().parent
stage=sys.argv[2]
assert stage in {'SMOKE','TRAIN','EVALUATE'}
ns=json.loads(Path(sys.argv[1]).read_text())
ns.update(STAGE=stage,USE_DRIVE=False,__name__='__main__')
files=['common','approve','gpu']+(['train'] if stage in {'SMOKE','TRAIN'} else ['eval_setup','scorer','evaluate'])
for name in files:
    p=SCRIPTS/(name+'.py'); exec(compile(p.read_text(),str(p),'exec'),ns)


## Execute stage
Read the actual outputs. A stopped or failed gate is not a completed stage.

In [ ]:
# @title Execute this notebook stage
def file_sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
def read_json(p): return json.loads(Path(p).read_text())
def save_json(p,x):
    p=Path(p); temp=p.with_suffix(p.suffix+'.tmp'); temp.write_text(json.dumps(x,indent=2)); temp.replace(p)
def record_controller(stage,status,detail):
    save_json(WORK/'auto_controller_state.json',{'stage':stage,'status':status,'detail':str(detail)})
def pause(stage,detail):
    record_controller(stage,'waiting',detail); print('\nPAUSED — '+stage+'\n'+str(detail)+'\nSave changes, then Runtime → Run all. Existing artifacts are preserved.',flush=True)
def cpu(stage, parts):
    ns={**CONFIG,'USE_DRIVE':False,'STAGE':stage,'__name__':'__main__'}
    for part in ['common']+parts:
        p=SCRIPT_DIR/(part+'.py'); exec(compile(p.read_text(),str(p),'exec'),ns)
    return ns

def gpu(stage):
    print('\n=== '+stage+' — isolated GPU worker ===',flush=True)
    record_controller(stage,'running','GPU subprocess')
    config_path=SCRIPT_DIR/'run_config.json'; save_json(config_path,CONFIG)
    env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'; env.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
    p=subprocess.Popen([sys.executable,'-u',str(SCRIPT_DIR/'worker.py'),str(config_path),stage],env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in p.stdout:
            print(line,end='',flush=True)
        status=p.wait()
    except BaseException:
        p.terminate()
        try: p.wait(timeout=20)
        except subprocess.TimeoutExpired: p.kill(); p.wait()
        raise
    if status: raise RuntimeError(f'{stage} worker exited with code {status}. Read the actual error above; checkpoints preserved.')
    # Child exit releases CUDA allocations. Dependencies already installed for subsequent child processes in this session.
    CONFIG['INSTALL_GPU_DEPS']=False

def validate_active(ns):
    approval,run,done,final=ns['active_run']()
    cfg=done['config']
    for k,v in {'base':BASE,'revision':REV,'max_length':MAX_LENGTH,'grad_accum':GRAD_ACCUM,'epochs':EPOCHS,'seed':SEED}.items():
        assert cfg[k]==v,f'Trained adapter uses different {k}; do not silently change a finished run.'
    assert done['steps']>0 and (final/'adapter_model.safetensors').is_file()
    return approval,run,done,final

def get_eval_artifacts(run):
    active=read_json(WORK/'active_evaluation.json'); assert active['run']==run.name
    folder=run/'evaluation'/active['evaluation']; return folder,read_json(folder/'evaluation_report.json')

def eval_receipt(run,done):
    folder,report=get_eval_artifacts(run)
    assert report['evaluation_config']['adapter_hashes']==done['adapter_hashes']
    assert report['evaluation_config']['fresh_hash']==FRESH_TEST_SHA256 and report['fresh']
    db=folder/'responses.sqlite3'
    with sqlite3.connect(db.resolve().as_uri()+'?mode=ro',uri=True) as conn:
        assert conn.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
        counts=dict(conn.execute('SELECT model,COUNT(*) FROM responses GROUP BY model'))
        expected=1000+report['fresh']['n']
        assert counts=={'qwen3_base':expected,'trimorious_v2':expected}, f'Incomplete paired evaluation: {counts}'
    return {'evaluation':folder.name,'report_sha256':file_sha(folder/'evaluation_report.json'),'db_sha256':file_sha(db),'mapping_sha256':file_sha(folder/'rubric_mapping_private.json'),'fresh_sha256':FRESH_TEST_SHA256}



def audit_stage(stage, paths):
    from datetime import datetime, timezone
    target=WORK/'stage_audits'; target.mkdir(exist_ok=True)
    files=[Path(p) for p in paths if Path(p).is_file()]
    for p in files: assert p.resolve().is_relative_to(WORK.resolve())
    manifest={str(p.relative_to(WORK)):file_sha(p) for p in files}
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    dest=target/f'{stage}_{stamp}_PRIVATE.zip'; temp=dest.with_suffix('.tmp')
    import zipfile
    with zipfile.ZipFile(temp,'w',zipfile.ZIP_DEFLATED) as z:
        for p in files: z.write(p,str(p.relative_to(WORK)))
        z.writestr('audit_manifest.json',json.dumps(manifest,indent=2))
    with zipfile.ZipFile(temp) as z:
        assert z.testzip() is None
        assert all(hashlib.sha256(z.read(k)).hexdigest()==v for k,v in manifest.items())
    temp.replace(dest); print('Private stage audit:',dest)

ns=cpu('RECOVER',['approve','eval_setup']); approval,run,done,final=validate_active(ns)
assert FRESH_TEST_PATH and FRESH_TEST_SHA256 and FRESH_INDEPENDENCE_ATTESTED, 'Supply independently authored fresh test and sealed hash first (see instructions).'
assert Path(FRESH_TEST_PATH).is_file() and file_sha(FRESH_TEST_PATH)==FRESH_TEST_SHA256, 'Missing fresh test or hash mismatch.'
assert not approval['fresh_test_sha256'] or approval['fresh_test_sha256']==FRESH_TEST_SHA256, 'Fresh test differs from frozen approval.'
d07=ROOT/D07_FILENAME
if not d07.exists():
    temp=Path('/content')/D07_FILENAME; assert temp.is_file(),f'Save original D07 ZIP to {ROOT}'
    import zipfile
    with zipfile.ZipFile(temp) as z: ns['archive_check'](z)
    pending=d07.with_suffix('.zip.tmp'); shutil.copyfile(temp,pending); assert file_sha(pending)==file_sha(temp); pending.replace(d07)
receipt_path=run/'auto_eval_receipt.json'
if receipt_path.exists():
    assert read_json(receipt_path)==eval_receipt(run,done), 'Completed evaluation artifacts changed.'
    print('Matching completed evaluation reused.')
else:
    gpu('EVALUATE'); save_json(receipt_path,eval_receipt(run,done))
folder,report=get_eval_artifacts(run)
audit_stage(STAGE,[WORK/'project_state.json',folder/'evaluation_report.json',receipt_path])
print('Objective checks:',report['checks'])
print('Review queue:',folder/'rubric_review_private.csv')
print('NEXT: notebook 06 after human rubric review.' if report['objective_gate_passed'] else 'STOP: evaluation failed release gates. Diagnose on development data; do not publish.')